# Preprocess and Train Pipeline
Run each cell sequentially to mirror the original Python pipeline while capturing intermediate files under `data/intermediate` and final outputs under `data/final`.
- Inputs are read from `Step3-Interpreting Production Scores` by default.
- File names match the original scripts; only folder placement changes.


In [39]:
from pathlib import Path

project_root = Path.cwd()
disease_name = input("Enter the disease name (A for Alzheimer's, B for Breast Cancer, C for Colon Cancer): ").strip().upper()

if disease_name == 'A':
    disease_folder = "Alzheimer"
elif disease_name == 'B':
    disease_folder = "Breast"
elif disease_name == 'C':
    disease_folder = "Colon"
else:
    raise ValueError("Invalid disease name. Please enter A, B, or C.")


input_dir = project_root / "data" / "input" / disease_folder
intermediate_dir = project_root / "data" / "intermediate" / disease_folder
final_dir = project_root / "data" / "final" / disease_folder

intermediate_dir.mkdir(parents=True, exist_ok=True)
final_dir.mkdir(parents=True, exist_ok=True)

score_data_path = input_dir / "ScoreData.mat"
metabolomics_path = input_dir / "metabolomics.csv"

print(f"Inputs: {score_data_path} and {metabolomics_path}")
print(f"Intermediate outputs -> {intermediate_dir}")
print(f"Final outputs -> {final_dir}")


Inputs: c:\Users\Lenovo\Desktop\23-24 Bahar\Bioinformatics\tamboor\graduation_final\data\input\Alzheimer\ScoreData.mat and c:\Users\Lenovo\Desktop\23-24 Bahar\Bioinformatics\tamboor\graduation_final\data\input\Alzheimer\metabolomics.csv
Intermediate outputs -> c:\Users\Lenovo\Desktop\23-24 Bahar\Bioinformatics\tamboor\graduation_final\data\intermediate\Alzheimer
Final outputs -> c:\Users\Lenovo\Desktop\23-24 Bahar\Bioinformatics\tamboor\graduation_final\data\final\Alzheimer


In [41]:
import scipy.io
import pandas as pd

def clear_metabolite_name(metabolite):
    return metabolite.split(' [')[0]

def extract_struct(mat_struct):
    data = {}
    for field in mat_struct._fieldnames:
        value = getattr(mat_struct, field)
        if isinstance(value, scipy.io.matlab.mio5_params.mat_struct):
            data[field] = extract_struct(value)
        else:
            data[field] = value
    return data

def write_sorted_excel(scores, names, path):
    df = pd.DataFrame({"Score": scores, "Metabolite": names})
    df_sorted = df.reindex(df["Score"].abs().sort_values().index)
    df_sorted.to_excel(path, index=False, header=False)
    print(f"Saved to {path}")

mat = scipy.io.loadmat(score_data_path, struct_as_record=False, squeeze_me=True)
score_data = extract_struct(mat["ScoreData"])

tamboor_scores = score_data["TAMBOORScore"][:, 3]
timbr_scores = score_data["TIMBRScore"][:, 2]
mod_timbr_scores = score_data["ModifiedTIMBRScore"][:, 2]

met_names = [clear_metabolite_name(str(m)) for m in score_data["metNames"]]

write_sorted_excel(tamboor_scores, met_names, intermediate_dir / "tamboor_results.xlsx")
write_sorted_excel(timbr_scores, met_names, intermediate_dir / "timbr_results.xlsx")
write_sorted_excel(mod_timbr_scores, met_names, intermediate_dir / "modified_timbr_results.xlsx")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17832\3618507817.py:11: DeprecationWarning: Please use `mat_struct` from the `scipy.io.matlab` namespace, the `scipy.io.matlab.mio5_params` namespace is deprecated.
  if isinstance(value, scipy.io.matlab.mio5_params.mat_struct):


KeyError: 'ModifiedTIMBRScore'

In [49]:
import requests
import pandas as pd
import json

def get_refmet_metabolites(params):
    refmet_url = "https://www.metabolomicsworkbench.org/databases/refmet/name_to_refmet_new_min.php"
    res = requests.post(refmet_url, data=params).text.split("\n")
    if res:
        res.pop(0)
    ref_dict = {}
    map_count = 0
    for line in res:
        if not line:
            continue
        met, ref, *_ = line.split("\t")
        if ref != "-":
            map_count += 1
            if ref_dict.get(ref) is None:
                ref_dict[ref] = met
            else:
                print(f"Different match for the same metabolite: {met} --> {ref} --> {ref_dict[ref]}")
        print(f"Metabolite: {met} Refmet: {ref}")
    print(f"Mapped {map_count} out of {len(res)}")
    return ref_dict

tamboor_file_path = intermediate_dir / "modified_timbr_results.xlsx"
metabolomics_file_path = metabolomics_path

tamboor_df = pd.read_excel(tamboor_file_path)
tamboor_metabolites = tamboor_df.iloc[:, 1].tolist()
tamboor_metabolites_string = ""

# Clear metabolite name from quotes and other parts
def clear_metabolite_name(metabolite):
    met = metabolite.split(' [')[0]
    met = met.split('\'')[1]
    return met

for i, met in enumerate(tamboor_metabolites):
    met = clear_metabolite_name(tamboor_metabolites[i])
    tamboor_metabolites[i] = met
    tamboor_metabolites_string += met + '\n'

metabolomics_df = pd.read_csv(metabolomics_file_path, nrows=1)
metabolomics_columns = metabolomics_df.columns.tolist()[6:]
metabolomics_string = ""

with open(intermediate_dir / "metabolomics_columns.txt", "w") as f:
    for col in metabolomics_columns:
        f.write(col + "\n")

with open(intermediate_dir / "tamboor_metabolites.txt", "w") as f:
    for met in tamboor_metabolites:
        f.write(met + "\n")

for col in metabolomics_columns:
    metabolomics_string += col + "\n"

data_params = {"metabolite_name": metabolomics_string}
params = {"metabolite_name": tamboor_metabolites_string}

print("Calling Metabolomics Workbench (requires network access)...")
ref_dict = get_refmet_metabolites(params)
ref_dict_metabolomics = get_refmet_metabolites(data_params)

with open(intermediate_dir / "tamboor_refmet.json", "w") as f:
    json.dump(ref_dict, f)

with open(intermediate_dir / "metabolomics_refmet.json", "w") as f:
    json.dump(ref_dict_metabolomics, f)


Calling Metabolomics Workbench (requires network access)...
Metabolite: Methionyl-Glutaminyl-Tyrosine Refmet: Met-Gln-Tyr
Metabolite: N-methylsalsolinol Refmet: N-Methylsalsolinol
Metabolite: 5,6-EET Refmet: 5(6)-EpETrE
Metabolite: Glutamyl-Tryptophanyl-Alanine Refmet: Glu-Trp-Ala
Metabolite: 13,16,19-docosatrienoic acid Refmet: 13,16,19-Docosatrienoic acid
Metabolite: Isoleucyl-Asparaginyl-Histidine Refmet: Ile-Asn-His
Metabolite: Glutamyl-Threonyl-Lysine Refmet: Glu-Thr-Lys
Metabolite: prostaglandin F1alpha Refmet: PGF1alpha
Metabolite: nicotinate Refmet: Nicotinic acid
Metabolite: 5(S)-HETE Refmet: 5S-HETE
Metabolite: Cis,Cis-11,14-Eicosadienoic Acid Refmet: -
Metabolite: galactose Refmet: Galactose
Metabolite: Glutamyl-Methioninyl-Histidine Refmet: -
Metabolite: Tryptophanyl-Prolyl-Leucine Refmet: Trp-Pro-Leu
Metabolite: (6Z,9Z,12Z,15Z,18Z)-TPA Refmet: -
Metabolite: Methionyl-Phenylalanyl-Arginine Refmet: Met-Phe-Arg
Metabolite: 9,10-hydroxyoctadec-12(Z)-enoate Refmet: -
Metabolite

In [50]:
import json

tamboor_wb_dict = {}
metabolomics_wb_dict = {}

with open(intermediate_dir / "tamboor_refmet.json") as f:
    tamboor_refmet_dict = json.load(f)

with open(intermediate_dir / "metabolomics_refmet.json") as f:
    metabolomics_refmet_dict = json.load(f)

print(f"Tamboor REFMET entries: {len(tamboor_refmet_dict)}")
print(f"Metabolomics REFMET entries: {len(metabolomics_refmet_dict)}")

metabolomics_wb_set = set(metabolomics_wb_dict.keys())
tamboor_wb_set = set(tamboor_wb_dict.keys())

metabolomics_refmet_set = set(metabolomics_refmet_dict.keys())
tamboor_refmet_set = set(tamboor_refmet_dict.keys())

tamboor_metabolite_results = set.union(tamboor_wb_set, tamboor_refmet_set)
metabolomics_metabolite_results = set.union(metabolomics_refmet_set, metabolomics_wb_set)

print(f"Tamboor metabolites: {len(tamboor_metabolite_results)}")
print(f"Metabolomics metabolites: {len(metabolomics_metabolite_results)}")

common_metabolites = tamboor_metabolite_results.intersection(metabolomics_metabolite_results)

print(f"Common metabolites total: {len(common_metabolites)}")

common_metabolites_refmet = tamboor_refmet_set.intersection(metabolomics_refmet_set)
common_metabolites_wb = tamboor_wb_set.intersection(metabolomics_wb_set)

print(f"Common metabolites for refmet mapping: {len(common_metabolites_refmet)}")
print(f"Common metabolites for wb mapping: {len(common_metabolites_wb)}")

common_metabolites_dict_tamboor = {}
for met in common_metabolites:
    if met in tamboor_wb_dict:
        common_metabolites_dict_tamboor[met] = tamboor_wb_dict[met]
    else:
        common_metabolites_dict_tamboor[met] = tamboor_refmet_dict[met]

with open(intermediate_dir / "common_metabolites_tamboor.json", "w") as f:
    json.dump(common_metabolites_dict_tamboor, f)

common_metabolites_dict_metabolomics = {}
for met in common_metabolites:
    if met in metabolomics_refmet_dict:
        common_metabolites_dict_metabolomics[met] = metabolomics_refmet_dict[met]
    else:
        common_metabolites_dict_metabolomics[met] = metabolomics_wb_dict[met]

with open(intermediate_dir / "common_metabolites_metabolomics.json", "w") as f:
    json.dump(common_metabolites_dict_metabolomics, f)


Tamboor REFMET entries: 756
Metabolomics REFMET entries: 563
Tamboor metabolites: 756
Metabolomics metabolites: 563
Common metabolites total: 170
Common metabolites for refmet mapping: 170
Common metabolites for wb mapping: 0


In [54]:
import pandas as pd
import json

def get_n_matched_metabolites(n):
    # Go backwards from the last metabolite in the tamboor_metabolites list
    metabolites_list = []
    half_matched_number = 0
    # Loop from len(tamboor_metabolites) - 1 to 0
    for i in range(len(tamboor_metabolites) - 1, -1, -1):
        print(f"Checking metabolite {i+1}/{len(tamboor_metabolites)}: {tamboor_metabolites[i]}")
        # If metaboliten name is not cleaned, clean it
        tamboor_metabolites[i] = clear_metabolite_name(tamboor_metabolites[i])

        if tamboor_metabolites[i] in common_metabolites_dict.keys():
            metabolites_list.append(common_metabolites_dict[tamboor_metabolites[i]])
            if i >= half:
                half_matched_number += 1
        # if abs(tamboor_values[i]) < 0.01:
        #     break
        # if len(metabolites_list) == n:
        #     break
    return metabolites_list, half_matched_number

algorithm_name = "tamboor"

excel_path = intermediate_dir / f"{algorithm_name}_results.xlsx"
metabolomics_file_path = metabolomics_path

tamboor_df = pd.read_excel(excel_path)
tamboor_values = tamboor_df.iloc[:, 0].tolist()
tamboor_metabolites = tamboor_df.iloc[:, 1].tolist()

half = len(tamboor_metabolites) // 2
print(half)

with open(intermediate_dir / "common_metabolites_tamboor.json") as f:
    common_metabolites_dict = json.load(f)

common_metabolites_dict = {v: k for k, v in common_metabolites_dict.items()}

metabolites_list, half_matched = get_n_matched_metabolites(100)

print("Length of metabolites list:", len(metabolites_list))
print("Half matched number:", half_matched)

with open(intermediate_dir / "common_metabolites_metabolomics.json") as f:
    common_metabolites_dict_metabolomics = json.load(f)

result_list = []
for met in metabolites_list:
    if met in common_metabolites_dict_metabolomics.keys():
        result_list.append(common_metabolites_dict_metabolomics[met])

print("Total mapped metabolites to metabolomics data:", len(result_list))

with open(intermediate_dir / "result_list.txt", "w") as f:
    for met in result_list:
        f.write(met + "\n")


517
Checking metabolite 1035/1035: 'GQ1b [Extracellular]'
Checking metabolite 1034/1035: 'fucosyl-galactosylgloboside [Extracellular]'
Checking metabolite 1033/1035: 'galfuc12gal14acglcgalgluside heparan sulfate [Extracellular]'
Checking metabolite 1032/1035: 'type IIIAb [Extracellular]'
Checking metabolite 1031/1035: 'Leb Glycolipid [Extracellular]'
Checking metabolite 1030/1035: 'V3(NeuAc)2-Gb5Cer [Extracellular]'
Checking metabolite 1029/1035: 'GD1beta [Extracellular]'
Checking metabolite 1028/1035: 'GD1c [Extracellular]'
Checking metabolite 1027/1035: 'fucfuc132galacglcgal14acglcgalgluside heparan sulfate [Extracellular]'
Checking metabolite 1026/1035: 'G00086 [Extracellular]'
Checking metabolite 1025/1035: 'Ley Glycolipid [Extracellular]'
Checking metabolite 1024/1035: 'fucfucfucgalacglcgal14acglcgalgluside heparan sulfate [Extracellular]'
Checking metabolite 1023/1035: 'lacto-N-fucopentaosyl-III-ceramide [Extracellular]'
Checking metabolite 1022/1035: 'galfucgalacglcgal14acglcgal

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import numpy as np

data = pd.read_csv(metabolomics_path)
if(disease_name == 'A'):
    target_column = "Diagnosis"
else:
    target_column = "Factors"

n = 40
with open(intermediate_dir / "result_list.txt", "r") as f:
    biomarkers = [next(f).strip() for _ in range(n)]

non_metabolite_columns = [target_column]

if disease_name == 'A':
    non_metabolite_columns = ["Diagnosis", "Sample ID", "Gender", "Race", "Braak"]

# drop columns that start with 'X -'
cols_to_remove = [col for col in data.columns if col.startswith('X -')]
data.drop(columns=cols_to_remove, inplace=True)


X_all = data.drop(columns=non_metabolite_columns)
X_biomarkers = data[biomarkers]

if disease_name == 'A':
    data[target_column] = data[target_column].replace('AD+', 'AD')

# only keep AD and Control samples for Alzheimer's
if disease_name == 'A':
    binary_classifications = {
        "Control vs.AD": ['Control', 'AD']
    }
# drop MCI samples for Alzheimer's
if disease_name == 'A':
    data = data[data[target_column] != 'MCI']


# if disease_name == 'A':
#     binary_classifications = {
#         "Control vs. MCI+AD": ['Control', 'MCI', 'AD']
#     }
else:
    binary_classifications = {
    "Control vs. Cancer": ['healthy', 'c']
}
    
print(f"Total number of samples in the dataset: {len(data)}")
print(f"Total number of features in the dataset: {data.shape[1] - len(non_metabolite_columns)}")
print(f"Number of biomarkers selected: {len(biomarkers)}")

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

with open(final_dir / "training_results.txt", "w") as output_file:
    for label, classes in binary_classifications.items():
        binary_data = data[data[target_column].isin(classes)].copy()

        if label == "Control vs. MCI+AD":
            binary_data[target_column] = binary_data[target_column].replace({'MCI': 'AD'})

        binary_data[target_column] = LabelEncoder().fit_transform(binary_data[target_column])

        y_binary = binary_data[target_column]
        X_all_binary = binary_data.drop(columns=non_metabolite_columns)
        X_biomarkers_binary = binary_data[biomarkers]

        random_features = np.random.choice(X_all.columns, n, replace=False).tolist()
        X_random_binary = binary_data[random_features]

        pipe_all = Pipeline([
            ('clf', RandomForestClassifier(random_state=42))
        ])
        pipe_biomarkers = Pipeline([
            ('clf', RandomForestClassifier(random_state=42))
        ])
        pipe_random = Pipeline([
            ('clf', RandomForestClassifier(random_state=42))
        ])

        accuracy_all = cross_val_score(pipe_all, X_all_binary, y_binary, cv=kf, scoring='accuracy').mean()
        precision_all = cross_val_score(pipe_all, X_all_binary, y_binary, cv=kf, scoring='precision_weighted').mean()
        recall_all = cross_val_score(pipe_all, X_all_binary, y_binary, cv=kf, scoring='recall_weighted').mean()
        f1_all = cross_val_score(pipe_all, X_all_binary, y_binary, cv=kf, scoring='f1_weighted').mean()

        accuracy_biomarkers = cross_val_score(pipe_biomarkers, X_biomarkers_binary, y_binary, cv=kf, scoring='accuracy').mean()
        precision_biomarkers = cross_val_score(pipe_biomarkers, X_biomarkers_binary, y_binary, cv=kf, scoring='precision_weighted').mean()
        recall_biomarkers = cross_val_score(pipe_biomarkers, X_biomarkers_binary, y_binary, cv=kf, scoring='recall_weighted').mean()
        f1_biomarkers = cross_val_score(pipe_biomarkers, X_biomarkers_binary, y_binary, cv=kf, scoring='f1_weighted').mean()

        accuracy_random = cross_val_score(pipe_random, X_random_binary, y_binary, cv=kf, scoring='accuracy').mean()
        precision_random = cross_val_score(pipe_random, X_random_binary, y_binary, cv=kf, scoring='precision_weighted').mean()
        recall_random = cross_val_score(pipe_random, X_random_binary, y_binary, cv=kf, scoring='recall_weighted').mean()
        f1_random = cross_val_score(pipe_random, X_random_binary, y_binary, cv=kf, scoring='f1_weighted').mean()

        print(f"Total number of samples in the dataset: {len(binary_data)}")
        print(f"Results for {label}:")
        print("Model with All Features:")
        print(f"Total Features: {X_all.shape[1]}")
        print(f"Accuracy: {accuracy_all:.4f}")
        print(f"Precision: {precision_all:.4f}")
        print(f"Recall: {recall_all:.4f}")
        print(f"F1 Score: {f1_all:.4f}")

        print("\nModel with Biomarkers:")
        print(f"Accuracy: {accuracy_biomarkers:.4f}")
        print(f"Precision: {precision_biomarkers:.4f}")
        print(f"Recall: {recall_biomarkers:.4f}")
        print(f"F1 Score: {f1_biomarkers:.4f}")

        print("\nModel with Randomly Selected Features:")
        print(f"Accuracy: {accuracy_random:.4f}")
        print(f"Precision: {precision_random:.4f}")
        print(f"Recall: {recall_random:.4f}")
        print(f"F1 Score: {f1_random:.4f}")
        print("\n" + "-"*50 + "\n")

        output_file.write(f"Results for {label}:\n")
        output_file.write("Model with All Features:\n")
        output_file.write(f"Accuracy: {accuracy_all:.4f}\n")
        output_file.write(f"Precision: {precision_all:.4f}\n")
        output_file.write(f"Recall: {recall_all:.4f}\n")
        output_file.write(f"F1 Score: {f1_all:.4f}\n")

        output_file.write("\nModel with Biomarkers:\n")
        output_file.write(f"Accuracy: {accuracy_biomarkers:.4f}\n")
        output_file.write(f"Precision: {precision_biomarkers:.4f}\n")
        output_file.write(f"Recall: {recall_biomarkers:.4f}\n")
        output_file.write(f"F1 Score: {f1_biomarkers:.4f}\n")

        output_file.write("\nModel with Randomly Selected Features:\n")
        output_file.write(f"Accuracy: {accuracy_random:.4f}\n")
        output_file.write(f"Precision: {precision_random:.4f}\n")
        output_file.write(f"Recall: {recall_random:.4f}\n")
        output_file.write(f"F1 Score: {f1_random:.4f}\n")
        output_file.write("\n" + "-"*50 + "\n\n")


Total number of samples in the dataset: 170
Total number of features in the dataset: 588
Number of biomarkers selected: 15
Total number of samples in the dataset: 164
Results for Control vs.AD:
Model with All Features:
Total Features: 588
Accuracy: 0.6710
Precision: 0.6786
Recall: 0.6710
F1 Score: 0.6619

Model with Biomarkers:
Accuracy: 0.8096
Precision: 0.8204
Recall: 0.8096
F1 Score: 0.8082

Model with Randomly Selected Features:
Accuracy: 0.5798
Precision: 0.5840
Recall: 0.5798
F1 Score: 0.5643

--------------------------------------------------

